In [1]:
import os
import json

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
MODEL_NAME = os.getenv("MODEL_NAME")

if not PROJECT_ENDPOINT:
    raise ValueError("PROJECT_ENDPOINT is missing from your .env file.")

if not MODEL_NAME:
    raise ValueError(
        "Set MODEL_NAME or MODEL_DEPLOYMENT_NAME in your .env file."
    )

credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)

foundry_client = project_client.get_openai_client()

print("Azure AI Foundry client initialized.")
print("Model deployment:", MODEL_NAME)

Azure AI Foundry client initialized.
Model deployment: gpt-5-mini


In [2]:
from mcp import Client

MCP_URL = "http://127.0.0.1:8001/mcp"

async with Client(MCP_URL) as mcp_client:
    discovery = await mcp_client.list_tools()

    for tool in discovery.tools:
        print("Name:", tool.name)
        print("Description:", tool.description)
        print("Input schema:", tool.input_schema)
        print("-" * 50)

Name: list_characters
Description: Retrieve all characters from the AOI character API.
Input schema: {'type': 'object', 'properties': {}, 'title': 'list_charactersArguments'}
--------------------------------------------------


In [3]:
async with Client(MCP_URL) as mcp_client:
    result = await mcp_client.call_tool("list_characters", {})

    print("Tool error:", result.is_error)

    for block in result.content:
        if hasattr(block, "text"):
            print(block.text)

Tool error: False
[
  {
    "id": 0,
    "name": "Olefer Greslund",
    "position": "Supreme commander",
    "personality": "Brave, strategic",
    "gender": "Male"
  },
  {
    "id": 1,
    "name": "Feernim Zatuuro",
    "position": "Mini-spacecraft pilot",
    "personality": "Shy but heroic, kind hearted",
    "gender": "Non-binary"
  },
  {
    "id": 2,
    "name": "Jonas Greslund",
    "position": "Spacecraft architect",
    "personality": "Inventive, detail-oriented",
    "gender": "Male"
  },
  {
    "id": 3,
    "name": "Nate Arziggio",
    "position": "Supreme commander",
    "personality": "Calm, assertive",
    "gender": "Male"
  },
  {
    "id": 4,
    "name": "Georgiy Krinlisov",
    "position": "Entertainer",
    "personality": "Goofy, cheerful",
    "gender": "Male"
  },
  {
    "id": 5,
    "name": "Kroznaar Evlon Bluurlu",
    "position": "Fitness coach",
    "personality": "Sporty, energetic, rough but caring",
    "gender": "Non-binary"
  },
  {
    "id": 6,
    "name

In [4]:
def mcp_result_to_text(result):
    """Extract readable text from an MCP tool result."""
    parts = [
        block.text
        for block in result.content
        if hasattr(block, "text")
    ]

    text_result = "\n".join(parts)

    if result.is_error:
        return "The MCP tool returned an error: " + text_result

    return text_result or str(result.content)


async def ask_with_mcp(question: str, max_tool_rounds: int = 5):
    async with Client(MCP_URL) as mcp_client:

        # 1. Discover the available MCP tools.
        discovery = await mcp_client.list_tools()

        discovered_tools = {
            tool.name: tool for tool in discovery.tools
        }

        if not discovered_tools:
            raise RuntimeError("The MCP server exposed no tools.")

        # 2. Convert MCP tool schemas into Foundry function tools.
        llm_tools = [
            {
                "type": "function",
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.input_schema,
                "strict": False,
            }
            for tool in discovery.tools
        ]

        print("Discovered tools:", list(discovered_tools))

        # 3. Send the question to the Foundry model.
        response = foundry_client.responses.create(
            model=MODEL_NAME,
            instructions=(
                "You are an assistant with access to MCP tools. "
                "For questions about the character dataset, use "
                "the available tools instead of guessing. "
                "Base your answers on the retrieved data."
            ),
            input=question,
            tools=llm_tools,
            tool_choice="auto",
        )

        # 4. Handle any tool calls requested by the model.
        for round_number in range(max_tool_rounds):

            calls = [
                item for item in response.output
                if item.type == "function_call"
            ]

            # No tool calls means the model has finished answering.
            if not calls:
                return response.output_text

            print(f"Tool round: {round_number + 1}")
            tool_outputs = []

            for call in calls:
                print("Requested tool:", call.name)

                if call.name not in discovered_tools:
                    output = f"Unknown MCP tool: {call.name}"

                else:
                    try:
                        arguments = json.loads(
                            call.arguments or "{}"
                        )

                        result = await mcp_client.call_tool(
                            call.name,
                            arguments,
                        )

                        output = mcp_result_to_text(result)

                    except Exception as exc:
                        output = (
                            f"Tool execution failed: "
                            f"{type(exc).__name__}: {exc}"
                        )

                # Every tool call must have a corresponding output.
                tool_outputs.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": output,
                })

            # 5. Return the tool results to the model.
            response = foundry_client.responses.create(
                model=MODEL_NAME,
                previous_response_id=response.id,
                input=tool_outputs,
                tools=llm_tools,
                tool_choice="auto",
            )

        return (
            "Maximum tool-call rounds reached. "
            "Try asking a more specific question."
        )

In [5]:
answer = await ask_with_mcp(
    "List the characters in my dataset. "
    "Use the MCP tool to retrieve the data."
)

print(answer)

Discovered tools: ['list_characters']
Tool round: 1
Requested tool: list_characters
Retrieved 7 characters via the MCP tool:

- id 0 — Olefer Greslund: Supreme commander; Brave, strategic; Male  
- id 1 — Feernim Zatuuro: Mini-spacecraft pilot; Shy but heroic, kind hearted; Non-binary  
- id 2 — Jonas Greslund: Spacecraft architect; Inventive, detail-oriented; Male  
- id 3 — Nate Arziggio: Supreme commander; Calm, assertive; Male  
- id 4 — Georgiy Krinlisov: Entertainer; Goofy, cheerful; Male  
- id 5 — Kroznaar Evlon Bluurlu: Fitness coach; Sporty, energetic, rough but caring; Non-binary  
- id 6 — Zeenula Pelphegopoulos: Heroine; Courageous, compassionate; Female

Would you like this exported to a file (CSV/JSON) or filtered/sorted in any way?


In [6]:
answer = await ask_with_mcp(
    "Which character has the most interesting background? "
    "Retrieve the character data first, then explain your choice "
    "using specific details from the dataset."
)

print(answer)

Discovered tools: ['list_characters']
Tool round: 1
Requested tool: list_characters
I retrieved the character data (IDs, names, positions, personalities, genders):

- 0 Olefer Greslund — Supreme commander — Brave, strategic — Male  
- 1 Feernim Zatuuro — Mini-spacecraft pilot — Shy but heroic, kind hearted — Non-binary  
- 2 Jonas Greslund — Spacecraft architect — Inventive, detail-oriented — Male  
- 3 Nate Arziggio — Supreme commander — Calm, assertive — Male  
- 4 Georgiy Krinlisov — Entertainer — Goofy, cheerful — Male  
- 5 Kroznaar Evlon Bluurlu — Fitness coach — Sporty, energetic, rough but caring — Non-binary  
- 6 Zeenula Pelphegopoulos — Heroine — Courageous, compassionate — Female

Which has the most interesting background: Feernim Zatuuro (ID 1).

Why (using specifics from the dataset)
- Position vs. personality creates a compelling tension: Feernim is a "mini-spacecraft pilot" (a role that implies technical skill, danger, and public responsibility) but is described as "shy